# 03 · Voting, validation-only threshold & Table 3 (cluster-aware CIs)  [CPU]
Threshold chosen on validation scans only; test evaluated PER SCAN with 95% CIs **bootstrapped by physical fruit** (honest under repeated measures).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()/'nbpkg'))
import numpy as np, pandas as pd
from config import CFG
import dataset as ds, eval_core as ec
CFG.out_dir.mkdir(parents=True, exist_ok=True)
print('data_root :', CFG.data_root); print('fruit_key :', CFG.fruit_key)

In [ ]:
import pickle
all_scores=pickle.load(open(CFG.out_dir/'scan_scores.pkl','rb'))
print('backbones:',list(all_scores))

## Step 1 — vote threshold on VALIDATION only

In [ ]:
thr={}
for bb,s in all_scores.items():
    t,o=ec.select_vote_threshold(s['val'],objective=CFG.threshold_objective); thr[bb]=t
    print(f'{bb:14s} threshold={t:3d} (val {CFG.threshold_objective}={o:.3f})')

## Step 2 — evaluate test with cluster-by-fruit bootstrap CIs

In [ ]:
res={bb:ec.evaluate_test(all_scores[bb]['test'],thr[bb],cluster=True,n_boot=CFG.n_boot)
     for bb in all_scores}
t3=ec.results_to_frame(res); display(t3)
t3.to_csv(CFG.out_dir/'table3.csv',index=False)
open(CFG.out_dir/'table3.md','w').write(t3.to_markdown(index=False))

## Step 3 — confusion matrices (per scan)

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,len(res),figsize=(4*len(res),3.4))
for a,(bb,r) in zip(np.atleast_1d(ax),res.items()):
    c=r['confusion']; M=np.array([[c['tn'],c['fp']],[c['fn'],c['tp']]])
    a.imshow(M,cmap='Blues')
    for (i,j),v in np.ndenumerate(M): a.text(j,i,v,ha='center',va='center')
    a.set_xticks([0,1]);a.set_xticklabels(['Ctrl','Inf']);a.set_yticks([0,1])
    a.set_yticklabels(['Ctrl','Inf']);a.set_title(bb)
plt.tight_layout();plt.savefig(CFG.out_dir/'confusion_matrices.png',dpi=150);plt.show()

Report as *point [95% CI]*; state single fruit-level split + validation-fixed threshold + cluster-by-fruit CIs in the Methods.